# Project: Gridworld value iteration 🗺️
Walk from the AAU campus to Jomfru Ane Gade on a slippery winter grid: actions sometimes slip sideways, the harbour is a −10 trap, and every step costs a little. Solve the MDP exactly with **value iteration** (Bellman optimality), verify the policy by solving the Bellman *expectation* equation as a linear system, and watch the policy change as you vary the discount and the slip probability.

Topic: [[RL Basics and MDPs]], [[Q-Learning and Policy Gradients]] · guide note: [[Project - Gridworld Value Iteration]]

**How this works:** cells marked **TODO** are yours. Run the ✅ check under each one; later cells stay skipped until the checks they need pass. Stuck? Open the hidden ▶ *Solution*. Data is downloaded once into `./data` (git-ignored).

In [ ]:
import numpy as np, matplotlib.pyplot as plt
DONE = set()
def check(name, test, hint):
    """Runs test(); ✅ if it returns True. Passed checks unlock the cells further down."""
    try:
        ok = bool(test())
    except NotImplementedError:
        print(f"❌ {name}: not implemented yet – fill in the TODO above."); return
    except Exception as e:
        print(f"❌ {name}: {type(e).__name__}: {e}"); return
    if ok: DONE.add(name); print(f"✅ {name}")
    else: print(f"❌ {name}: {hint}")

def ready(*names):
    missing = [n for n in names if n not in DONE]
    if missing: print("⏭  Skipped – pass these checks first:", ", ".join(missing))
    return not missing
GRID = ["S...#....",
        "..#.#.##.",
        "..#...#..",
        "....HHH..",
        ".#......G"]
ROWS, COLS = len(GRID), len(GRID[0]); ACTIONS = [(-1, 0), (0, 1), (1, 0), (0, -1)]; ARROWS = "↑→↓←"
cells = [(r, c) for r in range(ROWS) for c in range(COLS)]; sid = {rc: i for i, rc in enumerate(cells)}; n_states = len(cells)
def build(slip=0.2, step_cost=-0.04):
    P = np.zeros((n_states, 4, n_states)); R = np.zeros((n_states, 4))
    for (r, c), s in sid.items():
        ch = GRID[r][c]
        if ch in "GH": P[s, :, s] = 1; continue                    # terminal: absorbing, no more reward
        for a in range(4):
            for a2, pr in [(a, 1 - slip), ((a + 1) % 4, slip / 2), ((a - 1) % 4, slip / 2)]:
                nr, nc = r + ACTIONS[a2][0], c + ACTIONS[a2][1]
                if not (0 <= nr < ROWS and 0 <= nc < COLS) or GRID[nr][nc] == "#": nr, nc = r, c     # walls: stay
                s2 = sid[(nr, nc)]; P[s, a, s2] += pr
                R[s, a] += pr * (step_cost + {"G": 1.0, "H": -10.0}.get(GRID[nr][nc], 0.0))
    return P, R
P, R = build()
def show(V, pi, title):
    plt.figure(figsize=(7, 3.6)); plt.imshow(V.reshape(ROWS, COLS), cmap="RdYlGn"); plt.colorbar(); plt.title(title)
    for (r, c), s in sid.items():
        ch = GRID[r][c]; plt.text(c, r, ch if ch in "#GH" else ARROWS[pi[s]], ha="center", va="center", fontsize=13)
    plt.axis("off"); plt.show()

## 1. Value iteration\nRepeat $Q(s,a) = R(s,a) + \gamma\sum_{s'}P(s'|s,a)V(s')$ and $V(s)=\max_aQ(s,a)$ until the largest change is below `tol`. Return `(V, greedy policy)`. (`P @ V` does the sum over $s'$ for all $s,a$ at once.)

In [ ]:
def value_iteration(P, R, gamma=0.95, tol=1e-8, max_iter=10000):
    # TODO 1: Bellman optimality backups until convergence
    raise NotImplementedError  # TODO 1

## 2. Policy evaluation as a linear system\nFor a fixed policy the Bellman *expectation* equation $V = R_\pi + \gamma P_\pi V$ is linear: $V = (I - \gamma P_\pi)^{-1}R_\pi$. Return $V^\pi$. If value iteration is right, evaluating its policy gives back the same $V$.

In [ ]:
def evaluate_policy(P, R, policy, gamma=0.95):
    # TODO 2: solve (I − γ P_π) V = R_π
    raise NotImplementedError  # TODO 2

In [ ]:
def _t_vi():
    P2 = np.zeros((2, 1, 2)); P2[0, 0, 1] = 1; P2[1, 0, 1] = 1; R2 = np.array([[1.0], [0.0]])     # s0 → s1 (reward 1), s1 absorbing
    V, pi = value_iteration(P2, R2, gamma=0.9)
    return np.allclose(V, [1, 0]) and np.allclose(evaluate_policy(P2, R2, np.array([0, 0]), 0.9), [1, 0])
check("value iteration", _t_vi, "Two-state chain: V = [1, 0] from both methods.")
if ready("value iteration"):
    V, pi = value_iteration(P, R); Vpi = evaluate_policy(P, R, pi)
    show(V, pi, "optimal policy, slip 0.2, γ = 0.95")
    check("Bellman consistent", lambda: np.allclose(V, Vpi, atol=1e-6), "Evaluating the greedy policy must reproduce V*.")
    for slip, gamma in [(0.0, 0.95), (0.4, 0.95), (0.2, 0.5)]:
        P_, R_ = build(slip); V_, pi_ = value_iteration(P_, R_, gamma); show(V_, pi_, f"slip {slip}, γ = {gamma}")

**Look for:** with slip 0 the shortest path hugs the harbour; with slip 0.4 the agent takes the long safe way around. With γ = 0.5 the far-away goal is barely worth anything, so states far from G have values near 0 (or below, from step costs).

<details><summary>▶ Solution</summary>

```python
def value_iteration(P, R, gamma=0.95, tol=1e-8, max_iter=10000):
    V = np.zeros(P.shape[0])
    for _ in range(max_iter):
        Q = R + gamma * P @ V                  # (S, A)
        V_new = Q.max(1)
        if np.max(np.abs(V_new - V)) < tol:
            V = V_new
            break
        V = V_new
    return V, Q.argmax(1)
```

```python
def evaluate_policy(P, R, policy, gamma=0.95):
    idx = np.arange(P.shape[0])
    P_pi = P[idx, policy]; R_pi = R[idx, policy]
    return np.linalg.solve(np.eye(P.shape[0]) - gamma * P_pi, R_pi)
```
</details>

## Stretch goals
- Policy iteration: alternate `evaluate_policy` and greedy improvement. Count iterations vs. value iteration.
- Learn the same policy **without** knowing P and R: tabular Q-learning from sampled transitions ([[Q-Learning and Policy Gradients]], and the Snake agents in [[RL Arcade]]).
- Make the step cost 0 and γ = 1. What breaks?